In [1]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
folder_path = "C:/Users/nagal/OneDrive/Documents/Learn/Repo/Olist-Project/data"
data = {}
for file in os.listdir(folder_path):
    # print(file.split(".")[0])
    data[file.split(".")[0].replace("olist_","").replace("_dataset","").replace("order_","")] = folder_path + "/" + file
data

{'customers': 'C:/Users/nagal/OneDrive/Documents/Learn/Repo/Olist-Project/data/olist_customers_dataset.csv',
 'geolocation': 'C:/Users/nagal/OneDrive/Documents/Learn/Repo/Olist-Project/data/olist_geolocation_dataset.csv',
 'orders': 'C:/Users/nagal/OneDrive/Documents/Learn/Repo/Olist-Project/data/olist_orders_dataset.csv',
 'items': 'C:/Users/nagal/OneDrive/Documents/Learn/Repo/Olist-Project/data/olist_order_items_dataset.csv',
 'payments': 'C:/Users/nagal/OneDrive/Documents/Learn/Repo/Olist-Project/data/olist_order_payments_dataset.csv',
 'reviews': 'C:/Users/nagal/OneDrive/Documents/Learn/Repo/Olist-Project/data/olist_order_reviews_dataset.csv',
 'products': 'C:/Users/nagal/OneDrive/Documents/Learn/Repo/Olist-Project/data/olist_products_dataset.csv',
 'sellers': 'C:/Users/nagal/OneDrive/Documents/Learn/Repo/Olist-Project/data/olist_sellers_dataset.csv',
 'product_category_name_translation': 'C:/Users/nagal/OneDrive/Documents/Learn/Repo/Olist-Project/data/product_category_name_transla

In [ ]:
### Questions

80. Which payment methods are preferred by different states?

81. Which categories perform differently across regions?

82. Which combinations of category/payment/state produce interesting patterns?


In [ ]:
80. Which payment methods are preferred by different states?
I used pd.crosstab() to calculate payment counts for each state and payment type. 
    Then I used idxmax(axis=1) to identify the payment method with the highest count in each state.

In [22]:
state_payment.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,payment_sequential,payment_type,payment_installments,payment_value,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,1,credit_card,1,18.12,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP
1,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,3,voucher,1,2.00,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP
2,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,2,voucher,1,18.59,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP
3,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,1,boleto,1,141.46,af07308b275d755c9edb36a90c618231,47813,barreiras,BA
4,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,1,credit_card,3,179.12,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO


In [37]:
df_payments = pd.read_csv(data["payments"])
df_customers = pd.read_csv(data["customers"])
df_orders = pd.read_csv(data["orders"])
orders_payments = pd.merge(df_orders,df_payments,on="order_id",how="inner")
payments_customers = pd.merge(orders_payments,df_customers,on="customer_id",how="inner")
state_payment = pd.crosstab(
    index=payments_customers["customer_state"],
    columns=payments_customers["payment_type"]
)
state_payment


payment_type,boleto,credit_card,debit_card,not_defined,voucher
customer_state,,,,,
AC,16,61,2,0,5
AL,68,341,5,0,13
AM,21,124,2,0,7
AP,20,47,0,0,3
BA,614,2662,53,0,281
CE,205,1091,20,0,82
DF,396,1700,17,0,91
ES,403,1573,27,0,104
GO,449,1520,22,0,121


In [40]:
state_payment.idxmax(axis=1)

customer_state
AC    credit_card
AL    credit_card
AM    credit_card
AP    credit_card
BA    credit_card
CE    credit_card
DF    credit_card
ES    credit_card
GO    credit_card
MA    credit_card
MG    credit_card
MS    credit_card
MT    credit_card
PA    credit_card
PB    credit_card
PE    credit_card
PI    credit_card
PR    credit_card
RJ    credit_card
RN    credit_card
RO    credit_card
RR    credit_card
RS    credit_card
SC    credit_card
SE    credit_card
SP    credit_card
TO    credit_card
dtype: object

In [ ]:
81. Which categories perform differently across regions?

products -> product_category_name
items -> price
customers -> customer_state
translation -> product_category_name

I merged the customer, order, item, product, and category translation datasets to connect product revenue with customer states. 
    Then I used Pandas pivot_table() to calculate total revenue by state and product category. 
    I used idxmax() and max() to identify the top-performing category in each state and the top-performing state for each category. 
    Finally, I used sort_values(), value_counts(), and nlargest() to rank the results and identify business opportunities across regions.

  Function	                 Purpose

pd.merge()	---------> Combines related datasets using common keys
pivot_table() -------> Aggregates revenue across states and categories
idxmax(axis=1) ------> Returns the column name with the maximum value in each row
max(axis=1)	---------> Returns the maximum value in each row
sort_values() -------> Sorts results by revenue
value_counts() ------> Counts occurrences of each category or state
nlargest(5) ---------> Returns the five largest values

max() returns the maximum value, whereas idxmax() returns the index label where the maximum value occurs. 
In my Olist project, I used idxmax(axis=1) to identify the highest-revenue product category in each state and max(axis=1) to find its revenue.

In [46]:
df_products = pd.read_csv(data["products"])
df_products.head(1)

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0


In [47]:
df_items = pd.read_csv(data["items"])
df_items.head(1)

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.9,13.29


In [48]:
df_customers.head(1)

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP


In [49]:
df_orders.head(1)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00


In [62]:
df_product_category_name_translation = pd.read_csv(data["product_category_name_translation"])
df_product_category_name_translation.head(1)

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty


In [101]:
customers_orders = pd.merge(df_customers,df_orders,on="customer_id",how="inner") # states
customers_orders_items = pd.merge(customers_orders,df_items,on="order_id",how="inner") # revenue
category_region = pd.merge(customers_orders_items,df_products,on="product_id",how="inner") # category  
category_region_english = pd.merge(category_region,df_product_category_name_translation,on="product_category_name",how="inner") # english  

category_revenue = category_region_english.pivot_table(
    index = "customer_state",
    columns = "product_category_name_english",
    values = "price",
    aggfunc = "sum",
    fill_value = 0
)
#category_revenue.idxmax(axis=1)
#category_revenue.sort_values(ascending=False)
##################### Which category generates the most revenue in each state? #################################
top_category_revenue = pd.DataFrame({
    "top_category": category_revenue.idxmax(axis=1),
    "revenue": category_revenue.max(axis=1)
})

top_category_revenue = top_category_revenue.sort_values("revenue",ascending=False)
top_category_revenue.top_category.value_counts()
###################### Which states generate the most revenue for a specific category? ################################
state_revenue = category_region_english.pivot_table(
    index = "product_category_name_english",
    columns = "customer_state",
    values = "price",
    aggfunc = "sum",
    fill_value = 0
)


top_state_revenue = pd.DataFrame({
    "top_state": state_revenue.idxmax(axis=1),
    "revenue": state_revenue.max(axis=1)
})

top_state_revenue = top_state_revenue.sort_values("revenue",ascending=False)
#top_state_revenue.top_state.value_counts()
top_state_revenue["top_state"].value_counts()

########################### Top5 ##########################
# Top 5 states for health and beauty
state_revenue.loc["health_beauty"].sort_values(ascending=False)
state_revenue.loc["health_beauty"].nlargest(5)
# Top 5 categories in São Paulo
category_revenue.loc["SP"].sort_values(ascending=False)
category_revenue.loc["SP"].nlargest(5)


product_category_name_english
bed_bath_table           478284.52
health_beauty            462305.22
watches_gifts            435009.92
sports_leisure           386357.01
computers_accessories    350747.88
Name: SP, dtype: float64

In [ ]:
82. Which combinations of category/payment/state produce interesting patterns?
-> inner merge: Drops rows without matching category translations.
-> left merge + fillna("unknown_category"): Keeps those rows and labels missing translations.

for most of the payments are done by credit-card followed by boleto.
credit-card payments remain convenient and reliable at checkout

Total number of payments done by credit_card is 75487,76622
Total number of payments done by boleto is 19481,19766
Total number of payments done by voucher is  3736,3800
Total number of payments done by debit_card is  1501,1531
not_defined   =     3

    
for every type of payment SP state is having more number of payments count

SP is highest state having max_credit_card payments with max_payment_count 	31642,["after adding product_category_english to unknown_category"] = 32095
RR is lowest state having max_credit_card payments with max_payment_count 	31,["after adding product_category_english to unknown_category"] = 33


Q82: Which combinations of category, payment type, and state produce interesting patterns?

I analyzed payment behavior across product categories and customer states using Pandas merges and pivot tables. 
    Credit cards were the most frequently recorded payment type in my initial analysis, 
    and São Paulo showed the highest payment counts across payment types.

During data validation, 
-> Products whose category name is missing.
-> Products whose category name exists but has no English translation.
I identified 623 products without matching English category translations, including 610 with missing category names. 
    These missing translations can cause records to be excluded during inner joins. 
    I would preserve unmatched categories using a left join and validate the payment totals before finalizing the business findings.

This analysis demonstrates how payment patterns can be explored across regions and categories while accounting for data-quality limitations.


In [153]:
customers_orders = pd.merge(
    df_customers, df_orders,
    on="customer_id", how="inner"
)

orders_items = pd.merge(
    customers_orders, df_items,
    on="order_id", how="inner"
)

items_products = pd.merge(
    orders_items, df_products,
    on="product_id", how="inner"
)

category_english = pd.merge(
    items_products, df_product_category_name_translation,
    on="product_category_name", how="left"
)

category_english[
    "product_category_name_english"
] = category_english[
    "product_category_name_english"
].fillna("unknown_category")

category_payment_state = pd.merge(
    category_english[
        ["order_id", "customer_state",
         "product_category_name_english"]
    ],
    df_payments[
        ["order_id", "payment_type"]
        ],
    on="order_id",
    how="inner"
)
category_payment_pivot = category_payment_state.pivot_table(
    index=["customer_state","product_category_name_english"],
    columns="payment_type",
    values="order_id",
    aggfunc="nunique",
    fill_value=0
)
category_payment_pivot

payment_type                                  boleto  credit_card  debit_card  \
customer_state product_category_name_english                                    
AC             auto                                0            4           0   
               baby                                1            2           0   
               bed_bath_table                      0            3           0   
               books_general_interest              1            1           0   
               christmas_supplies                  1            0           0   
...                                              ...          ...         ...   
TO             stationery                          2            1           0   
               telephony                           7           13           0   
               toys                                2            9           1   
               unknown_category                    1            3           0   
               watches_gifts                       4           23           0   

payment_type                                  voucher  
customer_state product_category_name_english           
AC             auto                                 1  
               baby                                 0  
               bed_bath_table                       0  
               books_general_interest               0  
               christmas_supplies                   0  
...                                               ...  
TO             stationery                           0  
               telephony                            0  
               toys                                 0  
               unknown_category                     0  
               watches_gifts                        0  

[1382 rows x 4 columns]

In [154]:
# 4. Find the preferred payment type for each state-category combination
preferred_payment = category_payment_pivot.idxmax(axis=1)
preferred_payment.head(10)
preferred_payment_summary = pd.DataFrame({
    "preferred_payment": category_payment_pivot.idxmax(axis=1),
    "order_count": category_payment_pivot.max(axis=1)
})

preferred_payment_summary.head(10)

preferred_payment  order_count
customer_state product_category_name_english                               
AC             auto                                credit_card            4
               baby                                credit_card            2
               bed_bath_table                      credit_card            3
               books_general_interest                   boleto            1
               christmas_supplies                       boleto            1
               computers                           credit_card            1
               computers_accessories               credit_card            5
               consoles_games                           boleto            1
               cool_stuff                          credit_card            1
               electronics                         credit_card            3

In [155]:
print(category_payment_pivot.shape)
category_payment_pivot.head()

(1382, 4)


payment_type                                  boleto  credit_card  debit_card  \
customer_state product_category_name_english                                    
AC             auto                                0            4           0   
               baby                                1            2           0   
               bed_bath_table                      0            3           0   
               books_general_interest              1            1           0   
               christmas_supplies                  1            0           0   

payment_type                                  voucher  
customer_state product_category_name_english           
AC             auto                                 1  
               baby                                 0  
               bed_bath_table                       0  
               books_general_interest               0  
               christmas_supplies                   0

In [156]:
# 2. Which combinations have the most credit-card orders?
category_payment_pivot.sort_values(
    "credit_card", ascending=False
).head(10)


payment_type                                  boleto  credit_card  debit_card  \
customer_state product_category_name_english                                    
SP             bed_bath_table                    793         3499          65   
               health_beauty                     761         2900          71   
               sports_leisure                    701         2494          55   
               housewares                        473         2184          56   
               furniture_decor                   533         2100          38   
               computers_accessories             666         1918          64   
               watches_gifts                     373         1713          22   
               toys                              261         1306          20   
               auto                              309         1255          30   
RJ             bed_bath_table                    193         1145          15   

payment_type                                  voucher  
customer_state product_category_name_english           
SP             bed_bath_table                     173  
               health_beauty                      111  
               sports_leisure                     111  
               housewares                         140  
               furniture_decor                    138  
               computers_accessories               75  
               watches_gifts                       71  
               toys                                53  
               auto                                52  
RJ             bed_bath_table                      87

In [157]:
# 3. Compare payment methods for a specific state and category
category_payment_pivot.loc[
    ("SP", "health_beauty")
]


payment_type
boleto          761
credit_card    2900
debit_card       71
voucher         111
Name: (SP, health_beauty), dtype: int64

In [158]:
# 1.Which payment type is most common for each state-category combination?
category_payment_pivot.idxmax(axis=1)

customer_state  product_category_name_english
AC              auto                             credit_card
                baby                             credit_card
                bed_bath_table                   credit_card
                books_general_interest                boleto
                christmas_supplies                    boleto
                                                    ...     
TO              stationery                            boleto
                telephony                        credit_card
                toys                             credit_card
                unknown_category                 credit_card
                watches_gifts                    credit_card
Length: 1382, dtype: object

In [180]:
category_payment_pivot.sum().sort_values(ascending=False)

payment_type
credit_card    76622
boleto         19766
voucher         3800
debit_card      1531
dtype: int64

In [160]:
state_payment_counts = category_payment_pivot.groupby(
    level="customer_state"
).sum()

state_payment_counts

payment_type,boleto,credit_card,debit_card,voucher
customer_state,,,,
AC,16,62,2,4
AL,68,336,5,10
AM,21,123,2,4
AP,20,47,0,3
BA,613,2648,52,176
CE,206,1084,20,57
DF,392,1696,17,78
ES,406,1569,27,81
GO,450,1524,23,75


In [161]:
category_payment_pivot.head()

payment_type                                  boleto  credit_card  debit_card  \
customer_state product_category_name_english                                    
AC             auto                                0            4           0   
               baby                                1            2           0   
               bed_bath_table                      0            3           0   
               books_general_interest              1            1           0   
               christmas_supplies                  1            0           0   

payment_type                                  voucher  
customer_state product_category_name_english           
AC             auto                                 1  
               baby                                 0  
               bed_bath_table                       0  
               books_general_interest               0  
               christmas_supplies                   0

In [162]:
state_payment_counts["boleto"].idxmax()

'SP'

In [163]:
state_payment_counts["voucher"].idxmax()


'SP'

In [164]:
state_payment_summary = pd.DataFrame({
    "max_payment_type": state_payment_counts.idxmax(axis=1),
    "max_payment_count": state_payment_counts.max(axis=1)
})

state_payment_summary.sort_values(
    "max_payment_count", ascending=False
)

,max_payment_type,max_payment_count
customer_state,,
SP,credit_card,32095
RJ,credit_card,10288
MG,credit_card,9042
RS,credit_card,3993
PR,credit_card,3783
SC,credit_card,2713
BA,credit_card,2648
DF,credit_card,1696
ES,credit_card,1569


In [165]:
df_payments["payment_type"].value_counts(dropna=False)

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

In [166]:
len(df_payments)

103886

In [167]:
df_payments["payment_type"].value_counts(dropna=False).sum()

np.int64(103886)

In [168]:
orders_payments = pd.merge(
    df_orders,
    df_payments,
    on="order_id",
    how="inner"
)

print("Original payment records:", len(df_payments))
print("Merged payment records:", len(orders_payments))

orders_payments["payment_type"].value_counts(dropna=False)

Original payment records: 103886
Merged payment records: 103886


payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

In [169]:
print("Original payments:", len(df_payments))
print("Orders + payments:", len(orders_payments))
print("Category + payment rows:", len(category_payment_state))

print(
    "Payment types after category merge:"
)
print(
    category_payment_state["payment_type"]
    .value_counts(dropna=False)
)

Original payments: 103886
Orders + payments: 103886
Category + payment rows: 117601
Payment types after category merge:
payment_type
credit_card    86769
boleto         22867
voucher         6274
debit_card      1691
Name: count, dtype: int64


In [170]:
print(
    "Unique orders in orders + payments:",
    orders_payments["order_id"].nunique()
)

print(
    "Unique orders in category + payment:",
    category_payment_state["order_id"].nunique()
)

Unique orders in orders + payments: 99440
Unique orders in category + payment: 98665


In [171]:
orders_before_category_merge = set(
    orders_payments["order_id"]
)

orders_after_category_merge = set(
    category_payment_state["order_id"]
)

missing_orders = (
    orders_before_category_merge -
    orders_after_category_merge
)

print("Orders missing after category merge:", len(missing_orders))

Orders missing after category merge: 775


In [172]:
orders_payments[
    orders_payments["order_id"].isin(missing_orders)
].head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,payment_sequential,payment_type,payment_installments,payment_value
279,8e24261a7e58791d10cb1bf9da94df5c,64a254d30eed42cd0e6c36dddb88adf0,unavailable,2017-11-16 15:09:28,2017-11-16 15:26:57,NaN,NaN,2017-12-05 00:00:00,1,credit_card,5,84.00
614,c272bcd21c287498b4883c7512019702,9582c5bbecc65eb568e2c1d839b5cba1,unavailable,2018-01-31 11:31:37,2018-01-31 14:23:50,NaN,NaN,2018-02-16 00:00:00,1,credit_card,1,97.68
716,37553832a3a89c9b2db59701c357ca67,7607cd563696c27ede287e515812d528,unavailable,2017-08-14 17:38:02,2017-08-17 00:15:18,NaN,NaN,2017-09-05 00:00:00,1,boleto,1,132.46
766,d57e15fb07fd180f06ab3926b39edcd2,470b93b3f1cde85550fc74cd3a476c78,unavailable,2018-01-08 19:39:03,2018-01-09 07:26:08,NaN,NaN,2018-02-06 00:00:00,1,boleto,1,134.38
1175,00b1cb0320190ca0daa2c88b35206009,3532ba38a3fd242259a514ac2b6ae6b6,canceled,2018-08-28 15:26:39,NaN,NaN,NaN,2018-09-12 00:00:00,1,not_defined,1,0.00


In [173]:
missing_orders_df = orders_payments[
    orders_payments["order_id"].isin(missing_orders)
]

missing_order_items = pd.merge(
    missing_orders_df[["order_id"]],
    df_items,
    on="order_id",
    how="left"
)

print("Missing orders:", len(missing_orders))
print("Rows with no matching item:",
      missing_order_items["product_id"].isna().sum())

Missing orders: 775
Rows with no matching item: 830


In [174]:
missing_products = pd.merge(
    missing_order_items,
    df_products[["product_id", "product_category_name"]],
    on="product_id",
    how="left"
)

print(
    "Rows with no matching product:",
    missing_products["product_category_name"].isna().sum()
)

Rows with no matching product: 830


In [175]:
missing_translations = pd.merge(
    missing_products,
    df_product_category_name_translation,
    on="product_category_name",
    how="left"
)

print(
    "Rows with no English translation:",
    missing_translations[
        "product_category_name_english"
    ].isna().sum()
)

Rows with no English translation: 830


In [176]:
no_item_orders = missing_order_items.loc[
    missing_order_items["product_id"].isna(),
    "order_id"
].nunique()

print("Orders with no matching items:", no_item_orders)

Orders with no matching items: 775


In [177]:
items_without_products = df_items.loc[
    ~df_items["product_id"].isin(df_products["product_id"])
]

print("Item rows without matching products:",
      len(items_without_products))

print("Unique affected orders:",
      items_without_products["order_id"].nunique())

Item rows without matching products: 0
Unique affected orders: 0


In [178]:
untranslated_categories = df_products.loc[
    ~df_products["product_category_name"].isin(
        df_product_category_name_translation[
            "product_category_name"
        ]
    ),
    "product_category_name"
].drop_duplicates()

print(untranslated_categories)

105                                               NaN
1628                                         pc_gamer
5821    portateis_cozinha_e_preparadores_de_alimentos
Name: product_category_name, dtype: object


In [179]:
products_without_translation = df_products[
    ~df_products["product_category_name"].isin(
        df_product_category_name_translation[
            "product_category_name"
        ]
    )
]

print("Product rows without translation:",
      len(products_without_translation))

print("Unique product IDs:",
      products_without_translation["product_id"].nunique())

print("Missing category names:",
      products_without_translation["product_category_name"].isna().sum())

Product rows without translation: 623
Unique product IDs: 623
Missing category names: 610


In [181]:
df_payments["payment_type"].value_counts(dropna=False)

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

In [182]:
df_payments.groupby("payment_type")["order_id"].nunique()

payment_type
boleto         19784
credit_card    76505
debit_card      1528
not_defined        3
voucher         3866
Name: order_id, dtype: int64